# 사전학습: 재료 데이터를 읽고 조회하기
AI for Materials Science — Hands-on session 1

수업에서 바로 사용할 NumPy, pandas, pymatgen과 API 조회를 연습하겠습니다.
**기본 학습은 약 31분, 마지막 MP 선택 예제까지 하면 약 34분**입니다.
처음 패키지를 설치하거나 서버 응답을 기다리는 시간은 인터넷 환경에 따라 추가될 수 있습니다.

| 순서 | 할 일 | 학습 시간 |
|---|---|---:|
| 준비 | 환경과 데이터 준비 | 3분 |
| 1 | NumPy 배열과 조건 선택 | 6분 |
| 2 | CSV 읽기, 결측값 확인, 후보 선택 | 9분 |
| 3 | 화학식과 결정 구조, CIF 읽기·쓰기 | 8분 |
| 4 | 키 없이 OQMD 조회 | 5분 |
| 선택 | MP에서 LiFePO₄ 조회 | +3분 |

위에서부터 한 셀씩 실행해주세요. 코드의 `##` 줄은 설명용 주석입니다.
사전학습에서는 작은 예제 하나씩 따라가고, 자세한 MP 검색과 그래프 작성은 수업 시간에 이어가겠습니다.

## 준비: 라이브러리 설치와 GitHub 자료 받기

첫 셀에서 필요한 라이브러리를 설치하고, 둘째 셀에서 수업 저장소를 내려받겠습니다.
`%pip install`은 현재 노트북이 사용하는 환경에 라이브러리를 설치하는 명령입니다.

데이터는 저장소의 **`Hand-on-session1/Data/`**에 있습니다.
Colab에서 노트북만 열어도 `git clone`으로 CSV·CIF·OQMD 캐시를 함께 받을 수 있습니다.
이미 내려받은 저장소가 있으면 그대로 사용합니다. 개인 컴퓨터의 폴더 주소를 입력할 필요는 없습니다.

설치 후 재시작 안내가 나오면 커널/런타임을 재시작한 뒤 위에서부터 다시 실행해주세요.
실습에서 만든 결과는 현재 작업 폴더의 `outputs/01_preclass/`에 저장됩니다.

In [ ]:
## 사전학습에 필요한 라이브러리를 설치합니다. 데이터 조회에는 requests를 사용합니다.
%pip install -q numpy pandas pymatgen requests

In [ ]:
## 저장소를 내려받고, Data 폴더의 파일을 읽을 준비를 하겠습니다.
from pathlib import Path
from urllib.parse import parse_qs, urlparse
import os, json, hashlib
import numpy as np
import pandas as pd
import requests
from IPython.display import display

## Colab, 저장소 최상위 폴더, 실습 폴더에서 같은 노트북을 실행할 수 있습니다.
required_files = ["steel_strength.csv", "LiFePO4.cif",
                  "oqmd_li_fe_p_o_contains_sample.json", "sources.json"]
data_folders = [Path("Data"), Path("Hand-on-session1/Data"),
                Path("MS49900-AI4M/Hand-on-session1/Data"), Path("ai4m_data")]
DATA_DIR = next((folder for folder in data_folders
                 if all((folder / name).is_file() for name in required_files)), None)
if DATA_DIR is None:
    if not Path("MS49900-AI4M").exists():
        !git clone https://github.com/kwongibaek/MS49900-AI4M.git
    DATA_DIR = Path("MS49900-AI4M/Hand-on-session1/Data")
if not all((DATA_DIR / name).is_file() for name in required_files):
    raise FileNotFoundError("Data 파일이 없습니다. 수업 저장소를 정상적으로 내려받았는지 확인해주세요.")

## 파일 경로를 변수로 정해두면 뒤에서는 CSV와 CIF의 이름만 기억하면 됩니다.
STEEL_CSV = DATA_DIR / "steel_strength.csv"
LFP_CIF = DATA_DIR / "LiFePO4.cif"
OQMD_CACHE = DATA_DIR / "oqmd_li_fe_p_o_contains_sample.json"
OUTPUT_DIR = Path("outputs") / "01_preclass"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## SHA-256으로 배포 파일과 같은지 확인합니다. 원본 Data 파일은 직접 수정하지 마세요.
sources = json.loads((DATA_DIR / "sources.json").read_text(encoding="utf-8"))
for filename in required_files[:-1]:
    actual = hashlib.sha256((DATA_DIR / filename).read_bytes()).hexdigest()
    if actual != sources[filename]["sha256"]:
        raise ValueError(f"{filename}의 SHA-256이 배포본과 일치하지 않습니다.")

## OQMD가 느릴 때는 현재 검색 조건과 같은 실제 응답 사본만 사용합니다.
def load_matching_oqmd_cache(params):
    content = OQMD_CACHE.read_bytes()
    if hashlib.sha256(content).hexdigest() != sources[OQMD_CACHE.name]["sha256"]:
        raise ValueError("OQMD 캐시의 SHA-256이 배포본과 일치하지 않습니다.")
    payload = json.loads(content)
    saved_url = payload["meta"]["query"]["representation"]
    saved_params = parse_qs(urlparse(saved_url).query, keep_blank_values=True)
    if saved_params != {key: [str(value)] for key, value in params.items()}:
        raise RuntimeError("현재 OQMD 요청 조건이 캐시와 다릅니다. 네트워크를 연결하거나 원래 조건으로 되돌리세요.")
    return payload

print("데이터 폴더:", DATA_DIR)
print("준비가 끝났습니다. 1절부터 실행해보세요.")

## 1. NumPy: 배열에서 필요한 값 고르기 · 6분

항복강도 다섯 값을 배열에 담아보겠습니다. 배열은 여러 숫자를 한꺼번에 계산할 때 사용합니다.
여기서는 연습용 값을 쓰고, 다음 절에서 실제 실험 자료를 읽겠습니다.

In [ ]:
## np는 NumPy의 짧은 이름입니다. =로 오른쪽 배열을 왼쪽 변수에 저장합니다.
strength = np.array([620.0, 780.0, 910.0, 1050.0, 1180.0])
print("배열:", strength)
print("자료형과 모양:", strength.dtype, strength.shape)

## 위치는 0부터 셉니다. [1:4]는 위치 1, 2, 3을 고르며 끝 위치 4는 제외합니다.
print("첫 값:", strength[0])
print("마지막 값:", strength[-1])
print("일부 값:", strength[1:4])

In [ ]:
## 배열에 숫자 하나를 더하면 모든 값에 같은 숫자가 더해집니다.
adjusted = strength + 25.0
print("25 MPa를 더한 값:", adjusted)

## 비교 결과는 True/False 배열입니다. []에 넣으면 True인 위치만 남습니다.
mask = strength >= 900.0
selected = strength[mask]
print("900 MPa 이상인가요?", mask)
print("선택된 값:", selected)
print("선택된 개수와 평균:", selected.size, selected.mean())

## 2. pandas: CSV를 읽고 강재 후보 고르기 · 9분

`steel_strength.csv`에는 강재의 조성과 실험 물성이 들어 있습니다.
원소 함량은 wt%, 항복강도·인장강도는 MPa, 연신율은 %입니다.
`pd.read_csv()`로 읽은 표를 **DataFrame**이라고 부릅니다.

In [ ]:
## 준비 셀에서 받은 CSV를 읽습니다. 파일 경로에 개인 컴퓨터 주소를 넣지 않아도 됩니다.
steel = pd.read_csv(STEEL_CSV)
display(steel.head())

## shape는 (행 수, 열 수), len(DataFrame)은 행 수를 반환합니다.
print("행과 열:", steel.shape)
print("전체 행 수:", len(steel))

In [ ]:
## 여러 열을 고를 때는 열 이름의 리스트를 [] 안에 넣습니다.
property_columns = ["yield strength", "tensile strength", "elongation"]
properties = steel[property_columns]
display(properties.head())

## count()는 값이 있는 칸, isna().sum()은 비어 있는 칸을 열별로 셉니다.
display(pd.DataFrame({
    "값이 있는 개수": properties.count(),
    "결측값 개수": properties.isna().sum(),
}))

In [ ]:
## 강도와 연신율 조건을 따로 만듭니다. notna()는 값이 있는지 확인합니다.
strength_condition = steel["yield strength"] >= 1500.0
elongation_condition = steel["elongation"].notna() & (steel["elongation"] >= 10.0)

## &는 두 조건을 모두 만족해야 한다는 뜻입니다. loc[조건]으로 행을 고릅니다.
candidates = steel.loc[strength_condition & elongation_condition].copy()
candidates = candidates.sort_values("yield strength", ascending=False)
print("강도 조건 통과:", int(strength_condition.sum()))
print("연신율 조건 통과:", int(elongation_condition.sum()))
print("두 조건 모두 통과:", len(candidates))
display(candidates[["formula", *property_columns]].head())

## index=False는 DataFrame의 행 번호를 CSV의 별도 열로 저장하지 않는 설정입니다.
candidate_csv = OUTPUT_DIR / "steel_candidates.csv"
candidates.to_csv(candidate_csv, index=False)
print("저장한 파일:", candidate_csv)

## 3. pymatgen: 화학식과 결정 구조 다루기 · 8분

`Composition`은 원소의 종류와 개수를, `Structure`는 격자와 원자 위치까지 다룹니다.
먼저 LiFePO₄ 화학식을 읽고, 간단한 Si 구조를 만들어 CIF로 저장해보겠습니다.
마지막에는 준비된 LiFePO₄ CIF를 읽습니다.

실험 결정 구조를 찾을 때 ICSD는 일반적으로 이용 라이선스가 필요하고, COD는 공개 접근이 가능합니다.
여기서 읽는 CIF는 pymatgen에 공개된 예제 파일입니다.

In [ ]:
## Composition에 화학식을 넣으면 원소 종류와 개수를 해석해줍니다.
from pymatgen.core import Composition, Lattice, Structure

lfp_composition = Composition("LiFePO4")
print("간단한 화학식:", lfp_composition.reduced_formula)
print("화학식의 원자 수 합:", lfp_composition.num_atoms)

## 원자 분율은 해당 원소의 원자 수를 전체 원자 수로 나눈 값입니다.
print("원소별 원자 분율:", lfp_composition.fractional_composition.as_dict())

In [ ]:
## 공간군 대칭을 적용해 Si 원자들의 위치를 만듭니다. 격자 길이의 단위는 Å입니다.
silicon = Structure.from_spacegroup(
    "Fd-3m", Lattice.cubic(5.431), species=["Si"], coords=[[0, 0, 0]],
)
print("격자 길이:", silicon.lattice.abc)
print("이 셀의 원자 자리 수:", len(silicon))

## 구조 객체를 CIF 파일로 저장하면 다른 프로그램에서도 읽을 수 있습니다.
silicon_cif = OUTPUT_DIR / "silicon.cif"
silicon.to(filename=str(silicon_cif), fmt="cif")
print("저장한 구조:", silicon_cif)

In [ ]:
## from_file()은 CIF를 읽어 Structure 객체로 바꿉니다.
lfp_structure = Structure.from_file(LFP_CIF)
print("화학식:", lfp_structure.composition.reduced_formula)
print("격자 길이:", lfp_structure.lattice.abc)
print("원자 자리 수:", len(lfp_structure))

## [0]으로 첫 원자 자리를 꺼냅니다. 분율 좌표는 격자 벡터를 기준으로 한 좌표입니다.
print("첫 원소:", lfp_structure[0].species_string)
print("첫 원자의 분율 좌표:", lfp_structure[0].frac_coords)

## 4. OQMD: API 키 없이 계산 자료 조회하기 · 5분

API는 주소와 검색 조건을 보내면 프로그램이 읽을 수 있는 자료를 돌려줍니다.
OQMD에서 **Li, Fe, P, O를 모두 포함하는 자료를 최대 20건** 받아보겠습니다.
`element_set=(Li,Fe,P,O)`는 추가 원소도 허용합니다. 정확히 네 원소만 원하면 `AND ntypes=4`를 더합니다.

요청 → 응답 상태 확인 → JSON 읽기 → DataFrame 변환 순서를 보세요.
`delta_e`는 원자당 형성 에너지, `stability`는 convex hull 위 거리이며 둘 다 eV/atom 단위의 계산값입니다.

**서버가 느리면 실제로 받아두었던 캐시로 전환됩니다. 정상적인 처리이므로 표시된 출처를 확인하고 계속 진행하세요.**
같은 조건의 사본만 사용하며, 검색 조건을 바꾼 경우에는 다른 질의의 캐시를 대신 보여주지 않습니다.
OQMD와 MP의 에너지는 기준 에너지·구조·보정 방법을 맞추기 전에 행별로 직접 비교하면 안 됩니다.

In [ ]:
## params 딕셔너리에 검색 조건을 넣습니다. fields는 응답에서 가져올 항목입니다.
OQMD_FIELDS = ["name", "entry_id", "spacegroup", "ntypes", "natoms",
               "delta_e", "stability", "band_gap", "prototype", "icsd_id"]
OQMD_PARAMS = {
    "fields": ",".join(OQMD_FIELDS), "limit": 20, "offset": 0, "format": "json",
    "filter": "element_set=(Li,Fe,P,O)",
}
if os.getenv("AI4M_OFFLINE") == "1":
    oqmd_payload = load_matching_oqmd_cache(OQMD_PARAMS)
    oqmd_source = "OQMD 캐시: 오프라인 설정"
else:
    try:
        ## 실제 HTTP 요청을 보내고, 오류가 없으면 JSON을 Python 자료로 읽습니다.
        response = requests.get("https://oqmd.org/oqmdapi/formationenergy",
                                params=OQMD_PARAMS, timeout=30)
        response.raise_for_status()
        oqmd_payload = response.json()
        oqmd_source = "OQMD 실시간 HTTP 응답"
    except requests.RequestException as error:
        oqmd_payload = load_matching_oqmd_cache(OQMD_PARAMS)
        oqmd_source = f"OQMD 캐시: 요청 실패 ({type(error).__name__})"

## data의 목록을 표로 바꿉니다. 전체 검색 건수와 현재 페이지 행 수는 다를 수 있습니다.
if oqmd_payload.get("response_message") != "OK" or not isinstance(oqmd_payload.get("data"), list):
    raise ValueError("OQMD 응답의 상태 또는 data 형식을 확인해주세요.")
oqmd_table = pd.DataFrame(oqmd_payload["data"], columns=OQMD_FIELDS)
print("데이터 출처:", oqmd_source)
print("현재 페이지 행 수:", len(oqmd_table))
print("전체 검색 건수:", oqmd_payload.get("meta", {}).get("data_available"))
display(oqmd_table[["name", "entry_id", "ntypes", "delta_e", "stability"]])

## 선택: MP에서 LiFePO₄ 한 번 조회하기 · 3분

여기까지가 기본 사전학습입니다. MP API 키가 있는 학생은 같은 요청 방식을 한 번 더 써보세요.
`RUN_LIVE_MP=True`로 바꾸면 키를 화면에 표시하지 않고 입력받습니다.
키를 코드나 제출 파일에 직접 적지 마세요.

MP에서도 주소와 조건으로 자료를 요청할 수 있습니다. 같은 화학식에 구조가 여러 개라면 여러 행이 나옵니다.
이번에는 최대 10행만 확인하고, 자세한 검색 조건과 `MPRester` 사용법은 02 노트북에서 다루겠습니다.

In [ ]:
## API 키가 있고 선택 예제를 실행할 때만 True로 바꿔주세요.
from getpass import getpass
RUN_LIVE_MP = False
if RUN_LIVE_MP:
    mp_api_key = os.getenv("MP_API_KEY", "").strip() or getpass("MP API key: ").strip()
    if not mp_api_key:
        raise ValueError("MP API 키가 필요합니다.")
    MP_FIELDS = ["material_id", "formula_pretty", "band_gap", "energy_above_hull"]

    ## OQMD와 달리 MP에는 인증 키를 headers로 함께 보냅니다.
    mp_response = requests.get(
        "https://api.materialsproject.org/materials/summary/",
        headers={"X-API-KEY": mp_api_key},
        params={"formula": "LiFePO4", "_fields": ",".join(MP_FIELDS), "_limit": 10},
        timeout=30,
    )
    mp_response.raise_for_status()
    mp_rows = mp_response.json().get("data")
    if not isinstance(mp_rows, list):
        raise ValueError("MP 응답의 data 형식을 확인해주세요.")

    ## 응답 목록을 표로 바꾸어 밴드갭(eV)과 hull 위 에너지(eV/atom)를 확인합니다.
    mp_table = pd.DataFrame(mp_rows, columns=MP_FIELDS)
    display(mp_table)
    print("이번 요청에서 받은 행 수:", len(mp_table))
else:
    print("기본 사전학습을 마쳤습니다. MP 선택 예제는 건너뜁니다.")